<a href="https://colab.research.google.com/github/osmarbraz/sri/blob/main/3_2_1_GerarArquivosProjecaoEmbeddingsDocumento_v1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Gera os arquivos para o Embedding Projector dos documentos do conjunto de dados

**Entrada:** `dataset.zip` e `datasetpos.zip`.

- Dentro do arquivo compactado `dataset.zip` está o arquivo `dataset.csv`. Cada linha de `dataset.csv` é formado por `["id","sentencas","documento"]`.
   - `"id"` é o idenficador do documento na base de dados.
  - `"sentencas"` é uma lista com as sentenças do documento.
  - `"documento"` o documento limpo, mas não segmentado.

- Dentro do arquivo compactado `datasetpos.zip` está o arquivo `datasetpos.csv`. Cada linha do arquivo `datasetpos.csv` é formado por `["id","pos_documento"]`.
  - `"id"` é o idenficador do documento no dataset.
  - `"pos_documento"` é uma lista das sentenças do documento, formado por `"tokens","pos","verbos" e "lemma"`.
    - `"tokens"` é uma lista com os tokens da sentença.
    - `"pos"` é uma lista com as postagging das palavras da sentença.
    - `"verbos"` é uma lista com os verbos da sentença.
    - `"lemma"` é uma lista com os lemmas das palavras da sentença.

**Saída:** `projector_documento.zip`.

- Dentro do arquivo compactado `projector_documento.zip` estão os arquivos `records_documento.tsv` e  `meta_documento.tsv`. Cada linha do arquivo `records_documento.tsv` representa os embeddings de um documento. E cada linha do arquivo `meta_documento.tsv` os seus metadados.


**Processamento:**

1. Copia e descompacta o arquivo "`dataset.zip`" e "`datasetpos.zip`" para a máquina local do
Google Colab.
2. Gera os embeddings consolidados dos documentos utilizando o BERT. O notebook pode ser configurado para utilizar o BERTimbau **Large** e **Base**.
3. Gera os arquivos `records_documento.tsv` e `meta_documento.tsv` com os dados da projeção.
  3.1 O arquivo `records_documento.tsv` possui os registros dos embeddings consolidados de documentos. Pode ser configurado com as seguintes opções:    
    - O pooling dos embeddings dos documentos podem ser realizados com as estratégias Mean e Max.
    - Os embeddings podem ser recuperados da concatenação das 4 últimas camadas ou da última camada do BERT.    

  3.2 O arquivo `meta_documento.tsv` possui os metadados dos registros do arquivo anterior e possui as seguintes colunas:
    - `Documento`
    - `Id` (Id do documento)

4. Compacta os arquivos `records_documento.tsv` e `meta_documento.tsv` em `projector_documento.zip`.
5. Copia o arquivo `projector_documento.zip` para o google drive.
6. Exibe os arquivos da projeção no *Embedding Projector* embutido. Os arquivos podem ser feito o download para carregamento em *Embedding Projector* online em https://projector.tensorflow.org/.


---------------------------

Exemplo online da projeção dos embeddings:
https://projector.tensorflow.org/?config=https://raw.githubusercontent.com/osmarbraz/sri/main/projecao/config_documento.json

---------------------------

**Artigos complementares:**
- Artigo original BERT Jacob Devlin: https://arxiv.org/pdf/1506.06724.pdf

- Artigo do Embedding Projector: https://arxiv.org/pdf/1611.05469v1.pdf

- https://towardsdatascience.com/visualizing-bias-in-data-using-embedding-projector-649bc65e7487

- https://towardsdatascience.com/bert-visualization-in-embedding-projector-dfe4c9e18ca9

- https://krishansubudhi.github.io/deeplearning/2020/08/27/bert-embeddings-visualization.html

- https://amitness.com/interactive-sentence-embeddings/



# 1 Preparação do ambiente
Preparação do ambiente para execução do exemplo.

## 1.1 Tempo inicial de processamento

In [24]:
# Import das bibliotecas
import time
import datetime

#marca o tempo de início do processamento.
inicio_processamento = time.time()

## 1.2 Funções e classes auxiliares

Verifica se existe o diretório do notebook no diretório corrente.   


In [25]:
# Import das bibliotecas.
import os # Biblioteca para manipular arquivos

# ============================
def verificaDiretorioNotebook():
    '''
    Verifica se existe o diretório do notebook no diretório corrente.
    '''

    # Verifica se o diretório existe
    if not os.path.exists(DIRETORIO_NOTEBOOK):
        # Cria o diretório
        os.makedirs(DIRETORIO_NOTEBOOK)
        logging.info("Diretório do notebook criado: {}".format(DIRETORIO_NOTEBOOK))

    return DIRETORIO_NOTEBOOK

Remove tags de um documento

In [26]:
def remove_tags(documento):
  '''
  Remove tags de um documento
  '''

  import re

  documento_limpo = re.compile("<.*?>")
  return re.sub(documento_limpo, "", documento)

Função auxiliar para formatar o tempo como `hh: mm: ss`

In [27]:
# Import das bibliotecas.
import time
import datetime

def formataTempo(tempo):
  '''
  Pega a tempo em segundos e retorna uma string hh:mm:ss
  '''
  # Arredonda para o segundo mais próximo.
  tempo_arredondado = int(round((tempo)))

  # Formata como hh:mm:ss
  return str(datetime.timedelta(seconds=tempo_arredondado))

Classe(ModelosParametros) de definição dos parâmetros do modelo para medida

In [28]:
# Import das bibliotecas.
from dataclasses import dataclass, field
from typing import Dict, Optional
from typing import List

@dataclass
class ModelosParametros:
  max_seq_len: Optional[int] = field(
      default=None,
      metadata={'help': 'max seq len'},
  )
  pretrained_model_name_or_path: str = field(
      default='neuralmind/bert-base-portuguese-cased',
      metadata={'help': 'nome do modelo pré-treinado do BERT.'},
  )
  modelo_spacy: str = field(
      default="pt_core_news_lg",
      metadata={"help": "nome do modelo do spaCy."},
  )
  do_lower_case: bool = field(
      default=False,
      metadata={'help': 'define se o texto do modelo deve ser todo em minúsculo.'},
  )
  output_attentions: bool = field(
      default=False,
      metadata={'help': 'habilita se o modelo retorna os pesos de atenção.'},
  )
  output_hidden_states: bool = field(
      default=False,
      metadata={'help': 'habilita gerar as camadas ocultas do modelo.'},
  )

Biblioteca de limpeza de tela


In [29]:
# Import das bibliotecas.
from IPython.display import clear_output

## 1.3 Tratamento de logs

In [30]:
# Import das bibliotecas.
import logging # Biblioteca de logging

# Formatando a mensagem de logging
logging.basicConfig(format="%(asctime)s : %(levelname)s : %(message)s")

logger = logging.getLogger()
logger.setLevel(logging.INFO)

## 1.4  Identificando o ambiente Colab

In [31]:
# Se estiver executando no Google Colaboratory.
import sys

# Retorna true ou false se estiver no Google Colaboratory.
IN_COLAB = 'google.colab' in sys.modules

## 1.5 Colaboratory

Usando Colab GPU para Treinamento


Uma GPU pode ser adicionada acessando o menu e selecionando:

`Edit -> Notebook Settings -> Hardware accelerator -> (GPU)`

Em seguida, execute a célula a seguir para confirmar que a GPU foi detectada.

In [32]:
# Import das bibliotecas.
import tensorflow as tf

# Recupera o nome do dispositido da GPU.
device_name = tf.test.gpu_device_name()

# O nome do dispositivo deve ser parecido com o seguinte:
if device_name == "/device:GPU:0":
    logging.info("Encontrei GPU em: {}".format(device_name))
else:
    logging.info("Dispositivo GPU não encontrado")
    #raise SystemError("Dispositivo GPU não encontrado")

INFO:root:Encontrei GPU em: /device:GPU:0


Nome da GPU

Para que a torch use a GPU, precisamos identificar e especificar a GPU como o dispositivo. Posteriormente, em nosso ciclo de treinamento, carregaremos dados no dispositivo.

Vale a pena observar qual GPU você recebeu. A GPU Tesla P100 é muito mais rápido que as outras GPUs, abaixo uma lista ordenada:
- 1o Tesla P100
- 2o Tesla T4
- 3o Tesla P4 (Não tem memória para execução 4 x 8, somente 2 x 4)
- 4o Tesla K80 (Não tem memória para execução 4 x 8, somente 2 x 4)

In [33]:
# Import das bibliotecas.
import torch

def getDeviceGPU():
  '''
  Retorna um dispositivo de GPU se disponível ou CPU.

  Retorno:
    `device` - Um device de GPU ou CPU.
  '''

  # Se existe GPU disponível.
  if torch.cuda.is_available():

    # Diz ao PyTorch para usar GPU.
    device = torch.device("cuda")

    logging.info("Existem {} GPU(s) disponíveis.".format(torch.cuda.device_count()))
    logging.info("Iremos usar a GPU: {}.".format(torch.cuda.get_device_name(0)))

  # Se não.
  else:
    logging.info("Sem GPU disponível, usando CPU.")
    device = torch.device("cpu")

  return device

In [34]:
# Recupera o device com GPU ou CPU
device = getDeviceGPU()

INFO:root:Existem 1 GPU(s) disponíveis.
INFO:root:Iremos usar a GPU: Tesla T4.


Memória

Memória disponível no ambiente

In [35]:
# Importando as bibliotecas.
from psutil import virtual_memory

ram_gb = virtual_memory().total / 1e9
logging.info("Seu ambiente de execução tem {: .1f} gigabytes de RAM disponível\n".format(ram_gb))

if ram_gb < 20:
  logging.info("Para habilitar um tempo de execução de RAM alta, selecione menu o ambiente de execução> \"Alterar tipo de tempo de execução\"")
  logging.info("e selecione High-RAM. Então, execute novamente está célula")
else:
  logging.info("Você está usando um ambiente de execução de memória RAM alta!")

INFO:root:Seu ambiente de execução tem  13.6 gigabytes de RAM disponível

INFO:root:Para habilitar um tempo de execução de RAM alta, selecione menu o ambiente de execução> "Alterar tipo de tempo de execução"
INFO:root:e selecione High-RAM. Então, execute novamente está célula


## 1.6 Monta uma pasta no google drive para carregar os arquivos de dados.

In [36]:
# import necessário
from google.colab import drive

# Monta o drive na pasta especificada
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 1.7 Instalação do spaCy

https://spacy.io/

Modelos do spaCy para português:
https://spacy.io/models/pt

In [37]:
# Instala dependências do spacy
!pip install -U pip==25.3 setuptools==80.9.0 wheel==0.45.1

In [38]:
# Instala uma versão específica
!pip install -U spacy==3.8.11

## 1.8 Instalação do BERT

Instala a interface pytorch para o BERT by Hugging Face.

https://huggingface.co/docs/transformers/installation



In [39]:
!pip install -U transformers==4.49.0

# 2 Parametrização

## Gerais

In [40]:
#Definição dos parâmetros a serem avaliados

#Estratégia de recuperação dos embeddings: (0 - Embedding do token [CLS] da última,
#                                           1 - Embeddings da última camada,
#                                           2 - Embeddings da concatenação das 4 últimas camadas)
ESTRATEGIA_EMBEDDING = 0

# Estratégias para as palavras formadas por mais de um token do BERT (0 - Mean / 1 - Max)
ESTRATEGIA_POLLING = 0

Permite filtrar os documentos a serem utilizados na geração dos arquivos da projeção.

In [41]:
# Filtra um determinado documento
#FILTRO_DO = [] # Filtro vazio seleciona todos os documentos
#FILTRO_DO = ['1']
FILTRO_DO = []

## Específicos

Parâmetros do modelo

In [42]:
# Definição dos parâmetros do Modelo
model_args = ModelosParametros(
  max_seq_len = 512,
  #pretrained_model_name_or_path = "https://neuralmind-ai.s3.us-east-2.amazonaws.com/nlp/bert-large-portuguese-cased/bert-large-portuguese-cased_pytorch_checkpoint.zip",
  #pretrained_model_name_or_path = "https://neuralmind-ai.s3.us-east-2.amazonaws.com/nlp/bert-base-portuguese-cased/bert-base-portuguese-cased_pytorch_checkpoint.zip",

  #pretrained_model_name_or_path = "bert-large-cased",
  #pretrained_model_name_or_path = "bert-base-cased"
  #pretrained_model_name_or_path = "neuralmind/bert-large-portuguese-cased",
  pretrained_model_name_or_path = "neuralmind/bert-base-portuguese-cased",
  #pretrained_model_name_or_path = "bert-base-multilingual-cased",
  #pretrained_model_name_or_path = "bert-base-multilingual-uncased",

  #modelo_spacy = "en_core_web_lg",
  #modelo_spacy = "en_core_web_md",
  #modelo_spacy = "en_core_web_sm",
  modelo_spacy = "pt_core_news_lg",
  #modelo_spacy = "pt_core_news_md",
  #modelo_spacy = "pt_core_news_sm",

  do_lower_case = False,  # default True
  output_attentions = False,  # default False
  output_hidden_states = True, # default False
)

## Nome do diretório dos arquivos de dados

In [43]:
# Diretório do notebook
DIRETORIO_NOTEBOOK = "SRI"

## Define o caminho para os arquivos de dados

In [44]:
# Se estiver executando no Google Colaboratory
if IN_COLAB:

  # Diretório local para os arquivos de dados
  DIRETORIO_LOCAL = "/content/" + DIRETORIO_NOTEBOOK + "/"

  # Diretório no google drive com os arquivos de dados
  DIRETORIO_DRIVE = "/content/drive/MyDrive/Colab Notebooks/" + DIRETORIO_NOTEBOOK + "/data/"
else:

  # Diretório local para os arquivos de dados
  DIRETORIO_LOCAL = "./data/"

  # Diretório no google drive com os arquivos de dados
  DIRETORIO_DRIVE = "./data/"

## Inicialização diretórios

Diretório base local

In [45]:
# Importando as bibliotecas.
import os

# Se estiver executando no Google Colaboratory
if IN_COLAB:

  # Cria o diretório para receber os arquivos Originais e Permutados
  # Diretório a ser criado
  dirbase = DIRETORIO_LOCAL[:-1]

  if not os.path.exists(dirbase):
    # Cria o diretório
    os.makedirs(dirbase)
    logging.info("Diretório criado: {}.".format(dirbase))
  else:
    logging.info("Diretório já existe: {}.".format(dirbase))

INFO:root:Diretório já existe: /content/SRI.


# 3 spaCy

## 3.1 Download arquivo modelo

Uso:
https://spacy.io/usage

Modelos:
https://spacy.io/models

In [46]:
!python -m spacy download $model_args.modelo_spacy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 568.2/568.2 MB 24.8 MB/s  0:00:17
✔ Download and installation successful
You can now load the package via spacy.load('pt_core_news_lg')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


## 3.2 Carrega o modelo

In [47]:
# Import das bibliotecas.
import spacy # Biblioteca do spaCy

nlp = spacy.load(model_args.modelo_spacy)

## 3.3 Funções auxiliares spaCy

### getStopwords

Recupera as stopwords do spaCy

In [48]:
def getStopwords(nlp):
  '''
  Recupera as stop words do nlp(Spacy).

  Parâmetros:
    `nlp` - Um modelo spaCy carregado.
  '''

  spacy_stopwords = nlp.Defaults.stop_words

  return spacy_stopwords

Lista dos stopwords

In [49]:
logging.info("Quantidade de stopwords: {}.".format(len(getStopwords(nlp))))

print(getStopwords(nlp))

INFO:root:Quantidade de stopwords: 416.


{'cuja', 'lugar', 'fará', 'os', 'tiveram', 'atrás', 'ali', 'meses', 'enquanto', 'mil', 'fazeis', 'corrente', 'tendes', 'treze', 'a', 'baixo', 'sei', 'toda', 'dos', 'essa', 'ponto', 'então', 'só', 'logo', 'ademais', 'para', 'umas', 'mais', 'aquelas', 'fazem', 'tipo', 'pelos', 'tivestes', 'fazia', 'fim', 'desde', 'quem', 'custa', 'você', 'essas', 'vens', 'aqueles', 'talvez', 'seu', 'do', 'fazer', 'dão', 'na', 'quê', 'grandes', 'era', 'aos', 'põe', 'vindo', 'final', 'quanto', 'tua', 'fazes', 'inclusive', 'terceiro', 'após', 'fazemos', 'um', 'nas', 'daquele', 'estava', 'parte', 'meio', 'vinda', 'minhas', 'tive', 'oitava', 'estiveram', 'minha', 'daquela', 'nem', 'sim', 'irá', 'onde', 'dezassete', 'vossa', 'posso', 'muitos', 'dizem', 'cedo', 'nosso', 'nove', 'sem', 'dessa', 'obrigada', 'ambos', 'deverá', 'assim', 'algo', 'fora', 'cada', 'das', 'aquela', 'qual', 'apontar', 'faço', 'local', 'estás', 'quarto', 'te', 'mês', 'tudo', 'por', 'doze', 'dois', 'pôde', 'todos', 'nenhuma', 'grande', 'di

### getVerbos
Localiza os verbos da sentença

In [50]:
# Import das bibliotecas.
import spacy
from spacy.util import filter_spans
from spacy.matcher import Matcher

# (verbo normal como auxilar ou auxilar) + vários verbos auxiliares +verbo principal ou verbo auxiliar
gramaticav1 =  [
                {"POS": "AUX", "OP": "?", "DEP": {"IN": ["aux","aux:pass"]}},  #verbo auxiliar
                {"POS": "VERB", "OP": "?", "DEP": {"IN": ["ROOT","aux","xcomp","aux:pass"]}},  #verbo normal como auxiliar
                {"POS": "AUX", "OP": "*", "DEP": {"IN": ["aux","xcomp","aux:pass"]}},  #verbo auxiliar
                {"POS": "VERB", "OP": "+"}, #verbo principal
                {"POS": "AUX", "OP": "?", "DEP": {"IN": ["cop","aux","xcomp","aux:pass"]}},  #verbo auxiliar
               ]

# verbo auxiliar + verbo normal como auxiliar + conjunção com preposição + verbo
gramaticav2 =  [
                {"POS": "AUX", "OP": "?", "DEP": {"IN": ["aux","aux:pass"]}},  #verbo auxiliar
                {"POS": "VERB", "OP": "+", "DEP": {"IN": ["ROOT"]}},  #verbo principal
                {"POS": "SCONJ", "OP": "+", "DEP": {"IN": ["mark"]}}, #conjunção com preposição
                {"POS": "VERB", "OP": "+", "DEP": {"IN": ["xcomp"]}}, #verbo normal como complementar
               ]

#Somente verbos auxiliares
gramaticav3 =  [
                {"POS": "AUX", "OP": "?"},  #Verbos auxiliar
                {"POS": "AUX", "OP": "?", "DEP": {"IN": ["cop"]}},  #Verbos auxiliar de ligação (AUX+(cop))
                {"POS": "ADJ", "OP": "+", "DEP": {"IN": ["ROOT"]}},
                {"POS": "AUX", "OP": "?"}  #Verbos auxiliar
               ]

matcherv = Matcher(nlp.vocab)

matcherv.add("frase verbal", [gramaticav1])
matcherv.add("frase verbal", [gramaticav2])
matcherv.add("frase verbal", [gramaticav3])

#Retorna a Frase Verbal
def getVerbos(periodo):
  #Processa o período
  doc1 = nlp(periodo.text)

  # Chama o mather para encontrar o padrão
  matches = matcherv(doc1)

  padrao = [doc1[start:end] for _, start, end in matches]

  #elimina as repetições e sobreposições
  #return filter_spans(padrao)
  lista1 = filter_spans(padrao)

  # Converte os itens em string
  lista2 = []
  for x in lista1:
      lista2.append(str(x))

  return lista2

### getDicPOSQtde

Conta as POS Tagging de uma sentença

In [51]:
def getDicPOSQtde(sentenca):

    # Verifica se o sentenca não foi processado pelo spaCy
  if type(sentenca) is not spacy.tokens.doc.Doc:
      # Realiza o parsing no spacy
      doc = nlp(sentenca)
  else:
      doc = sentenca

  # Retorna inteiros que mapeiam para classes gramaticais
  conta_dicionarios = doc.count_by(spacy.attrs.IDS["POS"])

  # Dicionário com as tags e quantidades
  novodic = dict()

  for pos, qtde in conta_dicionarios.items():
    classe_gramatical = doc.vocab[pos].text
    novodic[classe_gramatical] = qtde

  return novodic

In [52]:
def getDicTodasPOSQtde(sentenca):

    # Verifica se o sentenca não foi processado pelo spaCy
  if type(sentenca) is not spacy.tokens.doc.Doc:
      # Realiza o parsing no spacy
      doc = nlp(sentenca)
  else:
      doc = sentenca

  # Retorna inteiros que mapeiam para classes gramaticais
  conta_dicionarios = doc.count_by(spacy.attrs.IDS["POS"])

  # Dicionário com as tags e quantidades
  novodic = {"PRON":0, "VERB":0, "PUNCT":0, "DET":0, "NOUN":0, "AUX":0, "CCONJ":0, "ADP":0, "PROPN":0, "ADJ":0, "ADV":0, "NUM":0, "SCONJ":0, "SYM":0, "SPACE":0, "INTJ":0, "X": 0}

  for pos, qtde in conta_dicionarios.items():
    classe_gramatical = doc.vocab[pos].text
    novodic[classe_gramatical] = qtde

  return novodic

### getDicTodasPOSQtde

Conta as POS Tagging de uma sentença

In [53]:
def getDicTodasPOSQtde(lista):

  # Dicionário com as tags e quantidades
  conjunto = {"PRON":0, "VERB":0, "PUNCT":0, "DET":0, "NOUN":0, "AUX":0, "CCONJ":0, "ADP":0, "PROPN":0, "ADJ":0, "ADV":0, "NUM":0, "SCONJ":0, "SYM":0, "SPACE":0, "INTJ": 0}

  for x in lista:
    valor = conjunto.get(x)
    if valor != None:
      conjunto[x] = valor + 1
    else:
      conjunto[x] = 1

  return conjunto

### getSomaDic

Soma os valores de dicionários com as mesmas chaves.

In [54]:
from collections import Counter
from functools import reduce

def atualizaValor(a,b):
    a.update(b)
    return a

def getSomaDic(lista):

  # Soma os dicionários da lista
  novodic = reduce(atualizaValor, (Counter(dict(x)) for x in lista))

  return novodic

### getTokensSentenca

Retorna a lista de tokens da sentenca.

In [55]:
def getTokensSentenca(sentenca):

    # Verifica se o sentenca não foi processado pelo spaCy
  if type(sentenca) is not spacy.tokens.doc.Doc:
      # Realiza o parsing no spacy
      doc = nlp(sentenca)
  else:
      doc = sentenca

  # Lista dos tokens
  lista = []

  # Percorre a sentença adicionando os tokens
  for token in doc:
    lista.append(token.text)

  return lista

### getPOSTokensSentenca

Retorna a lista das POS-Tagging dos tokens da sentenca.

In [56]:
def getPOSTokensSentenca(sentenca):

  # Verifica se o sentenca não foi processado pelo spaCy
  if type(sentenca) is not spacy.tokens.doc.Doc:
      # Realiza o parsing no spacy
      doc = nlp(sentenca)
  else:
      doc = sentenca

  # Lista dos tokens
  lista = []

  # Percorre a sentença adicionando os tokens
  for token in doc:
    lista.append(token.pos_)

  return lista

### getListaTokensPOSSentenca

Retorna duas listas uma com os tokens e a outra com a POS-Tagging dos tokens da sentenca.

In [57]:
def getListaTokensPOSSentenca(sentenca):
  # Verifica se o sentenca não foi processado pelo spaCy
  if type(sentenca) is not spacy.tokens.doc.Doc:
      # Realiza o parsing no spacy
      doc = nlp(sentenca)
  else:
      doc = sentenca

  # Lista dos tokens
  lista_tokens = []
  lista_pos = []

  # Percorre a sentença adicionando os tokens e as POS
  for token in doc:
    lista_tokens.append(token.text)
    lista_pos.append(token.pos_)

  return lista_tokens, lista_pos

### Tadução das tags

Tags de palavras universal

https://universaldependencies.org/u/pos/

Detalhes das tags em português:
http://www.dbd.puc-rio.br/pergamum/tesesabertas/1412298_2016_completo.pdf

In [58]:
#dicionário que contêm pos tag universal e suas explicações
palavra_universal_dict = {
  "X"    : "Outro",
  "VERB" : "Verbo ",
  "SYM"  : "Símbolo",
  "CONJ" : "Conjunção",
  "SCONJ": "Conjunção subordinativa",
  "PUNCT": "Pontuação",
  "PROPN": "Nome próprio",
  "PRON" : "Pronome substativo",
  "PART" : "Partícula, morfemas livres",
  "NUM"  : "Numeral",
  "NOUN" : "Substantivo",
  "INTJ" : "Interjeição",
  "DET"  : "Determinante, Artigo e pronomes adjetivos",
  "CCONJ": "Conjunção coordenativa",
  "AUX"  : "Verbo auxiliar",
  "ADV"  : "Advérbio",
  "ADP"  : "Preposição",
  "ADJ"  : "Adjetivo"
}

#Explica a POS
def getPOSPalavraUniversalTraduzido(palavra):
  if palavra in palavra_universal_dict.keys():
      traduzido = palavra_universal_dict[palavra]
  else:
      traduzido = "NA"
  return traduzido

### getSentencaSemStopWord

Retorna uma lista dos tokens sem as stopwords.

In [59]:
def getSentencaSemStopWord(sentenca, stopwords):

  # Lista dos tokens
  lista = []

  # Percorre os tokens da sentença
  for i, token in enumerate(sentenca):

    # Verifica se o token é uma stopword
    if token.lower() not in stopwords:
      lista.append(token)

  # Retorna o documento
  return lista

### getSentencaSalientePOS

Retorna uma lista das palavras do tipo especificado.

In [60]:
def getSentencaSalientePOS(sentenca, pos, tipo_saliente="NOUN"):

  # Lista dos tokens
  lista = []

  # Percorre a sentença
  for i, token in enumerate(sentenca):

    # Verifica se o token é do tipo especeficado
    if pos[i] == tipo_saliente:
      lista.append(token)

  # Retorna o documento
  return lista

###removeStopWords

Remove as stopwords de um documento ou senteça.

In [61]:
def removeStopWord1(documento, stopwords):

  # Remoção das stopwords do documento
  documento_sem_stopwords = [palavra for palavra in documento.split() if palavra.lower() not in stopwords]

  # Concatena o documento sem os stopwords
  documento_limpo = " ".join(documento_sem_stopwords)

  # Retorna o documento
  return documento_limpo

### retornaRelevante

Retorna somente os palavras do documento ou sentença do tipo especificado.

In [62]:
def retornaRelevante(documento, classe_relevante="NOUN"):

  # Corrigir!
  # Utilizar o documento já tokenizado pelo spacy!!!!
  # Existe uma lista com o documento e a sentença tokenizada pelo spacy

  # Realiza o parsing no spacy
  doc = nlp(documento)

  # Retorna a lista das palavras relevantes
  documento_com_substantivos = []
  for token in doc:
    #print("token:", token.pos_)
    if token.pos_ == classe_relevante:
      documento_com_substantivos.append(token.text)

  # Concatena o documento com os substantivos
  documento_concatenado = " ".join(documento_com_substantivos)

  # Retorna o documento
  return documento_concatenado

# 4 Carregando BERT Pre-Treinado

## 4.1 Carregando o modelo Pré-treinado BERT

Lista de modelos da comunidade:
* https://huggingface.co/models

Português(https://github.com/neuralmind-ai/portuguese-bert):  
* **"neuralmind/bert-base-portuguese-cased"**
* **"neuralmind/bert-large-portuguese-cased"**

In [63]:
# Import das bibliotecas
from transformers import BertModel

# Carrega o modelo
model = BertModel.from_pretrained(model_args.pretrained_model_name_or_path,
                                  output_attentions=model_args.output_attentions,
                                  output_hidden_states=model_args.output_hidden_states)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/647 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

## 4.2 Carregando o tokenizador BERT

O tokenizador utiliza WordPiece, veja em [artigo original](https://arxiv.org/pdf/1609.08144.pdf).

Por default(`do_lower_case=True`) todas as letras são colocadas para minúsculas. Para ignorar a conversão para minúsculo use o parâmetro `do_lower_case=False`. Esta opção também considera as letras acentuadas(ãçéí...), que são necessárias a língua portuguesa.

O parâmetro `do_lower_case` interfere na quantidade tokens a ser gerado a partir de um documento. Quando igual a `False` reduz a quantidade de tokens gerados.

In [64]:
# Import das bibliotecas
from transformers import BertTokenizer

# Carrega o tokenizador
#tokenizer = BertTokenizer.from_pretrained(model_args.pretrained_model_name_or_path)

tokenizer = BertTokenizer.from_pretrained(model_args.pretrained_model_name_or_path,
                                          do_lower_case=model_args.do_lower_case)

tokenizer_config.json:   0%|          | 0.00/43.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

## 4.3 Funções auxiliares do BERT

### getNomeModeloBERT

In [65]:
def getNomeModeloBERT(model_args):
  '''
  Recupera uma string com uma descrição do modelo BERT para nomes de arquivos e diretórios.

  Parâmetros:
  `model_args` - Objeto com os argumentos do modelo.

  Retorno:
  `MODELO_BERT` - Nome do modelo BERT.
  '''

  # Verifica o nome do modelo(default SEM_MODELO_BERT)
  MODELO_BERT = "SEM_MODELO_BERT"

  if 'neuralmind' in model_args.pretrained_model_name_or_path:
      MODELO_BERT = "_BERTimbau"

  else:
      if 'multilingual' in model_args.pretrained_model_name_or_path:
          MODELO_BERT = "_BERTmultilingual"

  return MODELO_BERT

### getTamanhoBERT

In [66]:
def getTamanhoBERT(model_args):
  '''
  Recupera uma string com o tamanho(dimensão) do modelo BERT para nomes de arquivos e diretórios.

  Parâmetros:
  `model_args` - Objeto com os argumentos do modelo.

  Retorno:
  `TAMANHO_BERT` - Nome do tamanho do modelo BERT.
  '''

  # Verifica o tamanho do modelo(default large)
  TAMANHO_BERT = "_large"

  if 'base' in model_args.pretrained_model_name_or_path:
      TAMANHO_BERT = "_base"

  return TAMANHO_BERT

### concatenaListas

In [67]:
def concatenaListas(lista, pos=1):
  lista_concat = []

  for x in lista:
      lista_concat = lista_concat + x[pos]

  return lista_concat

### getEmbeddingsCamadas

Funções que recuperam os embeddings das camadas:
- Primeira camada;
- Penúltima camada;
- Ùltima camada;
- Soma das 4 últimas camadas;
- Concatenação das 4 últimas camadas;
- Soma de todas as camadas.

In [68]:
def getEmbeddingPrimeiraCamada(output):
  # outputs[0] = last_hidden_state, outputs[1] = pooler_output, outputs[2] = hidden_states
  # hidden_states é uma lista python, e cada elemento um tensor pytorch no formado <lote> x <qtde_tokens> x <768 ou 1024>.

  # Retorna todas a primeira(-1) camada
  # Entrada: List das camadas(13 ou 25) (<1(lote)> x <qtde_tokens> <768 ou 1024>)
  resultado = output[2][0]
  # Saída: (<1(lote)> x <qtde_tokens> <768 ou 1024>)

  return resultado

def getEmbeddingPenultimaCamada(output):
  # outputs[0] = last_hidden_state, outputs[1] = pooler_output, outputs[2] = hidden_states
  # hidden_states é uma lista python, e cada elemento um tensor pytorch no formado <lote> x <qtde_tokens> x <768 ou 1024>.

  # Retorna todas a primeira(-1) camada
  # Entrada: List das camadas(13 ou 25) (<1(lote)> x <qtde_tokens> <768 ou 1024>)
  resultado = output[2][-2]
  # Saída: (<1(lote)> x <qtde_tokens> <768 ou 1024>)

  return resultado

def getEmbeddingUltimaCamada(output):
  # outputs[0] = last_hidden_state, outputs[1] = pooler_output, outputs[2] = hidden_states
  # hidden_states é uma lista python, e cada elemento um tensor pytorch no formado <lote> x <qtde_tokens> x <768 ou 1024>.

  # Retorna todas a primeira(-1) camada
  # Entrada: List das camadas(13 ou 25) (<1(lote)> x <qtde_tokens> <768 ou 1024>)
  resultado = output[2][-1]
  # Saída: (<1(lote)> x <qtde_tokens> <768 ou 1024>)

  return resultado

def getEmbeddingSoma4UltimasCamadas(output):
  # outputs[0] = last_hidden_state, outputs[1] = pooler_output, outputs[2] = hidden_states
  # hidden_states é uma lista python, e cada elemento um tensor pytorch no formado <lote> x <qtde_tokens> x <768 ou 1024>.

  # Retorna todas a primeira(-1) camada
  # Entrada: List das camadas(13 ou 25) (<1(lote)> x <qtde_tokens> <768 ou 1024>)
  embedding_camadas = output[2][-4:]
  # Saída: List das camadas(4) (<1(lote)> x <qtde_tokens> <768 ou 1024>)

  # Usa o método `stack` para criar uma nova dimensão no tensor
  # com a concateção dos tensores dos embeddings.
  #Entrada: List das camadas(4) (<1(lote)> x <qtde_tokens> <768 ou 1024>)
  resultado_stack = torch.stack(embedding_camadas, dim=0)
  # Saída: <4> x <1(lote)> x <qtde_tokens> x <768 ou 1024>

  # Realiza a soma dos embeddings de todos os tokens para as camadas
  # Entrada: <4> x <1(lote)> x <qtde_tokens> x <768 ou 1024>
  resultado = torch.sum(resultado_stack, dim=0)
  # Saida: <1(lote)> x <qtde_tokens> x <768 ou 1024>

  return resultado

def getEmbeddingConcat4UltimasCamadas(output):
  # outputs[0] = last_hidden_state, outputs[1] = pooler_output, outputs[2] = hidden_states
  # hidden_states é uma lista python, e cada elemento um tensor pytorch no formado <lote> x <qtde_tokens> x <768 ou 1024>.

  # Cria uma lista com os tensores a serem concatenados
  # Entrada: List das camadas(13 ou 25) (<1(lote)> x <qtde_tokens> x <768 ou 1024>)
  # Lista com os tensores a serem concatenados
  lista_concat = []

  # Percorre os 4 últimos
  for i in [-1,-2,-3,-4]:
      # Concatena da lista
      lista_concat.append(output[2][i])

  # Saída: Entrada: List das camadas(4) (<1(lote)> x <qtde_tokens> x <768 ou 1024>)
  # Realiza a concatenação dos embeddings de todos as camadas
  # Saída: Entrada: List das camadas(4) (<1(lote)> x <qtde_tokens> x <768 ou 1024>)
  resultado = torch.cat(lista_concat, dim=-1)

  # Saída: Entrada: (<1(lote)> x <qtde_tokens> x <3072 ou 4096>)
  return resultado

def getEmbeddingSomaTodasAsCamada(output):
  # outputs[0] = last_hidden_state, outputs[1] = pooler_output, outputs[2] = hidden_states
  # hidden_states é uma lista python, e cada elemento um tensor pytorch no formado <lote> x <qtde_tokens> x <768 ou 1024>.

  # Retorna todas as camadas descontando a primeira(0)
  # Entrada: List das camadas(13 ou 25) (<1(lote)> x <qtde_tokens> <768 ou 1024>)
  embedding_camadas = output[2][1:]
  # Saída: List das camadas(12 ou 24) (<1(lote)> x <qtde_tokens> <768 ou 1024>)

  # Usa o método `stack` para criar uma nova dimensão no tensor
  # com a concateção dos tensores dos embeddings.
  #Entrada: List das camadas(12 ou 24) (<1(lote)> x <qtde_tokens> <768 ou 1024>)
  resultado_stack = torch.stack(embedding_camadas, dim=0)
  # Saída: <12 ou 24> x <1(lote)> x <qtde_tokens> x <768 ou 1024>

  # Realiza a soma dos embeddings de todos os tokens para as camadas
  # Entrada: <12 ou 24> x <1(lote)> x <qtde_tokens> x <768 ou 1024>
  resultado = torch.sum(resultado_stack, dim=0)
  # Saida: <1(lote)> x <qtde_tokens> x <768 ou 1024>

  return resultado

### getEmbeddingsVisual

Função para gerar as coordenadas de plotagem a partir das sentenças de embeddings.

Existe uma função para os tipos de camadas utilizadas:
- Ùltima camada;
- Soma das 4 últimas camadas;
- Concatenação das 4 últimas camadas;
- Soma de todas as camadas.

In [69]:
# Import das bibliotecas
import numpy as np

def getEmbeddingsVisualUltimaCamada(documento, modelo, tokenizer):

  # Adiciona os tokens especiais
  documento_marcado = "[CLS] " + documento + " [SEP]"

  # Divide a sentença em tokens
  documento_tokenizado = tokenizer.tokenize(documento_marcado)

  # Mapeia as strings dos tokens em seus índices do vocabuário
  tokens_indexados = tokenizer.convert_tokens_to_ids(documento_tokenizado)

  # Marca cada um dos tokens como pertencentes à sentença "1".
  mascara_atencao = [1] * len(documento_tokenizado)

  # Converte a entrada em tensores
  tokens_tensores = torch.as_tensor([tokens_indexados])
  mascara_atencao_tensores = torch.as_tensor([mascara_atencao])

  # Prediz os atributos dos estados ocultos para cada camada
  with torch.no_grad():
      # Retorno de model quando ´output_hidden_states=True´ é setado:
      #outputs[0] = last_hidden_state, outputs[1] = pooler_output, outputs[2] = hidden_states
      outputs = modelo(tokens_tensores, mascara_atencao_tensores)

  # Camada embedding
  camada = getEmbeddingUltimaCamada(outputs)

  # Remove a dimensão 1, o lote "batches".
  token_embeddings = torch.squeeze(camada, dim=0)

  # Recupera os embeddings dos tokens como um vetor
  embeddings = token_embeddings.numpy()

  # Converte para um array
  W = np.array(embeddings)
  # Transforma em um array
  B = np.array([embeddings[0], embeddings[-1]])
  # Invertee B.T
  Bi = np.linalg.pinv(B.T)

  #Projeta a palavra no espaço
  Wp = np.matmul(Bi,W.T)

  return Wp, documento_tokenizado

In [70]:
# Import das bibliotecas
import numpy as np

def getEmbeddingsVisualSoma4UltimasCamadas(documento, modelo, tokenizer):

  # Adiciona os tokens especiais
  documento_marcado = "[CLS] " + documento + " [SEP]"

  # Divide a sentença em tokens
  documento_tokenizado = tokenizer.tokenize(documento_marcado)

  # Mapeia as strings dos tokens em seus índices do vocabuário
  tokens_indexados = tokenizer.convert_tokens_to_ids(documento_tokenizado)

  # Marca cada um dos tokens como pertencentes à sentença "1".
  mascara_atencao = [1] * len(documento_tokenizado)

  # Converte a entrada em tensores
  tokens_tensores = torch.as_tensor([tokens_indexados])
  mascara_atencao_tensores = torch.as_tensor([mascara_atencao])

  # Prediz os atributos dos estados ocultos para cada camada
  with torch.no_grad():
      # Retorno de model quando ´output_hidden_states=True´ é setado:
      #outputs[0] = last_hidden_state, outputs[1] = pooler_output, outputs[2] = hidden_states
      outputs = modelo(tokens_tensores, mascara_atencao_tensores)

  # Camada embedding
  camada = getEmbeddingSoma4UltimasCamadas(outputs)

  # Remove a dimensão 1, o lote "batches".
  token_embeddings = torch.squeeze(camada, dim=0)

  # Recupera os embeddings dos tokens como um vetor
  embeddings = token_embeddings.numpy()

  # Converte para um array
  W = np.array(embeddings)
  # Transforma em um array
  B = np.array([embeddings[0], embeddings[-1]])
  # Invertee B.T
  Bi = np.linalg.pinv(B.T)

  #Projeta a palavra no espaço
  Wp = np.matmul(Bi,W.T)

  return Wp, documento_tokenizado

In [71]:
# Import das bibliotecas
import numpy as np

def getEmbeddingsVisualConcat4UltimasCamadas(documento, modelo, tokenizer):

  # Adiciona os tokens especiais
  documento_marcado = "[CLS] " + documento + " [SEP]"

  # Divide a sentença em tokens
  documento_tokenizado = tokenizer.tokenize(documento_marcado)

  # Mapeia as strings dos tokens em seus índices do vocabuário
  tokens_indexados = tokenizer.convert_tokens_to_ids(documento_tokenizado)

  # Marca cada um dos tokens como pertencentes à sentença "1".
  mascara_atencao = [1] * len(documento_tokenizado)

  # Converte a entrada em tensores
  tokens_tensores = torch.as_tensor([tokens_indexados])
  mascara_atencao_tensores = torch.as_tensor([mascara_atencao])

  # Prediz os atributos dos estados ocultos para cada camada
  with torch.no_grad():
      # Retorno de model quando ´output_hidden_states=True´ é setado:
      #outputs[0] = last_hidden_state, outputs[1] = pooler_output, outputs[2] = hidden_states
      outputs = modelo(tokens_tensores, mascara_atencao_tensores)

  # Camada embedding
  camada = getEmbeddingConcat4UltimasCamadas(outputs)

  # Remove a dimensão 1, o lote "batches".
  token_embeddings = torch.squeeze(camada, dim=0)

  # Recupera os embeddings dos tokens como um vetor
  embeddings = token_embeddings.numpy()

  # Converte para um array
  W = np.array(embeddings)
  # Transforma em um array
  B = np.array([embeddings[0], embeddings[-1]])
  # Invertee B.T
  Bi = np.linalg.pinv(B.T)

  #Projeta a palavra no espaço
  Wp = np.matmul(Bi,W.T)

  return Wp, documento_tokenizado

In [72]:
# Import das bibliotecas
import numpy as np

def getEmbeddingsVisualSomaTodasAsCamadas(documento, modelo, tokenizer):

  # Adiciona os tokens especiais
  documento_marcado = "[CLS] " + documento + " [SEP]"

  # Divide a sentença em tokens
  documento_tokenizado = tokenizer.tokenize(documento_marcado)

  # Mapeia as strings dos tokens em seus índices do vocabuário
  tokens_indexados = tokenizer.convert_tokens_to_ids(documento_tokenizado)

  # Marca cada um dos tokens como pertencentes à sentença "1".
  mascara_atencao = [1] * len(documento_tokenizado)

  # Converte a entrada em tensores
  tokens_tensores = torch.as_tensor([tokens_indexados])
  mascara_atencao_tensores = torch.as_tensor([mascara_atencao])

  # Prediz os atributos dos estados ocultos para cada camada
  with torch.no_grad():
      # Retorno de model quando ´output_hidden_states=True´ é setado:
      #outputs[0] = last_hidden_state, outputs[1] = pooler_output, outputs[2] = hidden_states
      outputs = modelo(tokens_tensores, mascara_atencao_tensores)

  # Camada embedding
  camada = getEmbeddingSomaTodasAsCamada(outputs)

  # Remove a dimensão 1, o lote "batches".
  token_embeddings = torch.squeeze(camada, dim=0)

  # Recupera os embeddings dos tokens como um vetor
  embeddings = token_embeddings.numpy()

  # Converte para um array
  W = np.array(embeddings)
  # Transforma em um array
  B = np.array([embeddings[0], embeddings[-1]])
  # Invertee B.T
  Bi = np.linalg.pinv(B.T)

  #Projeta a palavra no espaço
  Wp = np.matmul(Bi,W.T)

  return Wp, documento_tokenizado

### getEmbeddings

Função para gerar os embeddings de sentenças.

Existe uma função para os tipos de camadas utilizadas:
- Ùltima camada;
- Soma das 4 últimas camadas;
- Concatenação das 4 últimas camadas;
- Soma de todas as camadas.

In [73]:
# Import das bibliotecas
import numpy as np

def getEmbeddingsUltimaCamada(documento, modelo, tokenizer):

  # Adiciona os tokens especiais
  documento_marcado = "[CLS] " + documento + " [SEP]"

  # Divide a sentença em tokens
  documento_tokenizado = tokenizer.tokenize(documento_marcado)

  # Mapeia as strings dos tokens em seus índices do vocabuário
  tokens_indexados = tokenizer.convert_tokens_to_ids(documento_tokenizado)

  # Marca cada um dos tokens como pertencentes à sentença "1".
  mascara_atencao = [1] * len(documento_tokenizado)

  # Converte a entrada em tensores
  tokens_tensores = torch.as_tensor([tokens_indexados])
  mascara_atencao_tensores = torch.as_tensor([mascara_atencao])

  # Prediz os atributos dos estados ocultos para cada camada
  with torch.no_grad():
      # Retorno de model quando ´output_hidden_states=True´ é setado:
      #outputs[0] = last_hidden_state, outputs[1] = pooler_output, outputs[2] = hidden_states
      outputs = modelo(tokens_tensores, mascara_atencao_tensores)

  # Camada embedding
  camada = getEmbeddingUltimaCamada(outputs)

  # Remove a dimensão 1, o lote "batches".
  token_embeddings = torch.squeeze(camada, dim=0)

  return token_embeddings, documento_tokenizado

In [74]:
# Import das bibliotecas
import numpy as np

def getEmbeddingsSoma4UltimasCamadas(documento, modelo, tokenizer):

  # Adiciona os tokens especiais
  documento_marcado = "[CLS] " + documento + " [SEP]"

  # Divide a sentença em tokens
  documento_tokenizado = tokenizer.tokenize(documento_marcado)

  # Mapeia as strings dos tokens em seus índices do vocabuário
  tokens_indexados = tokenizer.convert_tokens_to_ids(documento_tokenizado)

  # Marca cada um dos tokens como pertencentes à sentença "1".
  mascara_atencao = [1] * len(documento_tokenizado)

  # Converte a entrada em tensores
  tokens_tensores = torch.as_tensor([tokens_indexados])
  mascara_atencao_tensores = torch.as_tensor([mascara_atencao])

  # Prediz os atributos dos estados ocultos para cada camada
  with torch.no_grad():
      # Retorno de model quando ´output_hidden_states=True´ é setado:
      #outputs[0] = last_hidden_state, outputs[1] = pooler_output, outputs[2] = hidden_states
      outputs = modelo(tokens_tensores, mascara_atencao_tensores)

  # Camada embedding
  camada = getEmbeddingSoma4UltimasCamadas(outputs)

  # Remove a dimensão 1, o lote "batches".
  token_embeddings = torch.squeeze(camada, dim=0)

  return token_embeddings, documento_tokenizado

In [75]:
# Import das bibliotecas
import numpy as np

def getEmbeddingsConcat4UltimasCamadas(documento, modelo, tokenizer):
  # Adiciona os tokens especiais
  documento_marcado = "[CLS] " + documento + " [SEP]"

  # Divide a sentença em tokens
  documento_tokenizado = tokenizer.tokenize(documento_marcado)

  # Mapeia as strings dos tokens em seus índices do vocabuário
  tokens_indexados = tokenizer.convert_tokens_to_ids(documento_tokenizado)

  # Marca cada um dos tokens como pertencentes à sentença "1".
  mascara_atencao = [1] * len(documento_tokenizado)

  # Converte a entrada em tensores
  tokens_tensores = torch.as_tensor([tokens_indexados])
  mascara_atencao_tensores = torch.as_tensor([mascara_atencao])

  # Prediz os atributos dos estados ocultos para cada camada
  with torch.no_grad():
      # Retorno de model quando ´output_hidden_states=True´ é setado:
      #outputs[0] = last_hidden_state, outputs[1] = pooler_output, outputs[2] = hidden_states
      outputs = modelo(tokens_tensores, mascara_atencao_tensores)

  # Camada embedding
  camada = getEmbeddingConcat4UltimasCamadas(outputs)

  # Remove a dimensão 1, o lote "batches".
  token_embeddings = torch.squeeze(camada, dim=0)

  return token_embeddings, documento_tokenizado

In [76]:
# Import das bibliotecas
import numpy as np

def getEmbeddingsSomaTodasAsCamadas(documento, modelo, tokenizer):

  # Adiciona os tokens especiais
  documento_marcado = "[CLS] " + documento + " [SEP]"

  # Divide a sentença em tokens
  documento_tokenizado = tokenizer.tokenize(documento_marcado)

  # Mapeia as strings dos tokens em seus índices do vocabuário
  tokens_indexados = tokenizer.convert_tokens_to_ids(documento_tokenizado)

  # Marca cada um dos tokens como pertencentes à sentença "1".
  mascara_atencao = [1] * len(documento_tokenizado)

  # Converte a entrada em tensores
  tokens_tensores = torch.as_tensor([tokens_indexados])
  mascara_atencao_tensores = torch.as_tensor([mascara_atencao])

  # Prediz os atributos dos estados ocultos para cada camada
  with torch.no_grad():
      # Retorno de model quando ´output_hidden_states=True´ é setado:
      #outputs[0] = last_hidden_state, outputs[1] = pooler_output, outputs[2] = hidden_states
      outputs = modelo(tokens_tensores, mascara_atencao_tensores)

  # Camada embedding
  camada = getEmbeddingSomaTodasAsCamada(outputs)

  # Remove a dimensão 1, o lote "batches".
  token_embeddings = torch.squeeze(camada, dim=0)

  return token_embeddings, documento_tokenizado

### getDocumentoTokenizado

Retorna o documento tokenizado

In [77]:
def getDocumentoTokenizado(documento, tokenizer):
  '''
  Retorna o documento tokenizado pelo BERT.

  Parâmetros:
    `documento` - Documento a ser tokenizado.
    `tokenizer` - Tokenizador do BERT.
  '''

  # Adiciona os tokens especiais.
  documento_marcado = "[CLS] " + documento + " [SEP]"

  # Documento tokenizado
  documento_tokenizado = tokenizer.tokenize(documento_marcado)

  del tokenizer

  return documento_tokenizado

### encontrarIndiceSubLista

Retorna os índices de início e fim da sublista na lista

In [78]:
def encontrarIndiceSubLista(lista: List, sublista: List):
  '''
  Localiza os índices de início e fim de uma sublista em uma lista.
  Baseado no algoritmo de https://codereview.stackexchange.com/questions/19627/finding-sub-list
  de  https://en.wikipedia.org/wiki/Boyer%E2%80%93Moore%E2%80%93Horspool_algorithm

  Parâmetros:
    `lista` - Uma lista.
    `sublista` - Uma sublista a ser localizada na lista.

  Retorno:
    Os índices de início e fim da sublista na lista.
  '''
  # Tamanho da lista
  h = len(lista)
  # Tamanho da sblista
  n = len(sublista)
  # Cria um dicionário com os saltos descrescentes dos elementos n-1 da sublista
  skip = {sublista[i]: n - i - 1 for i in range(n - 1)}
  i = n - 1
  # Percorre a lista
  while i < h:
    # Percorre a sublista
    for j in range(n):
      # Se elemento da lista diferente da sublista pula a interação
      if lista[i - j] != sublista[-j - 1]:
        # Passa para o próximo elemento da lista saltando a sublista
        i += skip.get(lista[i], n)
        # Interrompe o for.
        break
    else:
      #Finalizando a pesquisa depois de executar todo o for(sem break)
      indice_inicio = i - n + 1
      indice_fim = indice_inicio + len(sublista)-1
      # Retorna o início e fim da sublista na lista
      return indice_inicio, indice_fim

  # Não encontrou a sublista na lista
  return -1, -1

### getEmbeddingSentencaEmbeddingDocumentoComTodasPalavras

A partir dos embeddings do documento, localiza o indíce de início e fim de uma sentença no documento e retorna os embeddings da sentença.

In [79]:
def getEmbeddingSentencaEmbeddingDocumentoComTodasPalavras(embedding_documento,
                                                           token_BERT_documento,
                                                           sentenca,
                                                           tokenizer):

  # Tokeniza a sentença
  sentenca_tokenizada_BERT = getDocumentoTokenizado(sentenca, tokenizer)
  #print(sentenca_tokenizada_BERT)

  # Remove os tokens de início e fim da sentença
  sentenca_tokenizada_BERT.remove("[CLS]")
  sentenca_tokenizada_BERT.remove("[SEP]")
  #print(len(sentenca_tokenizada_BERT))

  # Localiza os índices dos tokens da sentença no documento
  inicio, fim = encontrarIndiceSubLista(token_BERT_documento, sentenca_tokenizada_BERT)
  #print(inicio,fim)

  # Recupera os embeddings dos tokens da sentença a partir dos embeddings do documento
  embedding_sentenca = embedding_documento[inicio:fim+1]
  #print("embedding_sentenca=", embedding_sentenca.shape)

  del tokenizer
  del token_BERT_documento
  del embedding_documento

  # Retorna o embedding da sentença no documento
  return embedding_sentenca, sentenca_tokenizada_BERT

### getEmbeddingDocumentoComTodasPalavrasMean

In [80]:
# Importa a biblioteca
import torch

def getEmbeddingDocumentoComTodasPalavrasMean(embedding_documento):
  '''
  Calcula a média dos embeddings do documento excluindo os tokens
  especiais [CLS] do início e [SEP] do fim.
  Remove primeira dimensão devido ao cálculo da média.

  Parâmetros:
    `embedding_documento` - Embedding do documento.
  '''

  # Calcula a média dos embeddings para os tokens de embedding_documento, removendo a primeira dimensão.
  # Entrada: <qtde_tokens> x <768 ou 1024>
  #print("embedding_documento1=", embedding_documento.shape)
  media_embedding_documento = torch.mean(embedding_documento[1:-1], dim=0)
  # Saída: <768 ou 1024>

  del embedding_documento

  return media_embedding_documento

### getEmbeddingDocumentoRelevanteMean

In [81]:
# Importa a biblioteca
import torch

def getEmbeddingDocumentoRelevanteMean(id_documento,
                                       index_sentenca,
                                       embedding_documento,
                                       token_BERT_documento,
                                       documento,
                                       token_documento,
                                       pos_documento,
                                       filtro):
  '''
  Calcula a média dos embeddings do documento considerando tokens do tipo especificado no filtro.
  Remove primeira dimensão devido ao cálculo da média.

  Parâmetros:
    `embedding_documento` - Embeddings do documento gerados pelo BERT.
    `token_BERT_documento` - Lista com os tokens do documento gerados pelo tokenizador BERT.
    `documento` - Texto com o documento.
    `tokenizer` - Tokenizador do BERT.
    `token_documento` - Lista com os tokens do documento.
    `pos_documento` - Lista com as POS-Tagging do documento.
    `filtro` - Filtro dos embeddings.

  '''

  # Recupera a lista de tokens do documento, a lista dos postagging e a lista dos seus embeddings com um mesmo tamanho
  lista_tokens, lista_postagging, lista_embeddings = getTokensEmbeddingsPOSSentenca(id_documento,
                                                                                    index_sentenca,
                                                                                    embedding_documento,
                                                                                    token_BERT_documento,
                                                                                    documento,
                                                                                    token_documento,
                                                                                    pos_documento)

  #print("len(token_BERT_documento):", len(token_BERT_documento))
  #print("token_BERT_documento:", token_BERT_documento)
  #print("len(pos_documento):", len(pos_documento))
  #print("pos_documento:", pos_documento)
  #print("filtro:", filtro)
  #print()

  # Lista com os tensores selecionados
  lista_tokens_selecionados = []
  # Localizar os embeddings dos tokens da sentença tokenizada sem stop word no documento
  for i, token_documento in enumerate(lista_tokens):
      if (lista_postagging[i] in filtro):
          #print("Adicionando palavra do embedding:", lista_tokens[i])
          lista_tokens_selecionados.append(lista_embeddings[i])

  if  len(lista_tokens_selecionados) != 0:
      # Empila os embeddings da lista pela dimensão 0
      embedding_relevante = torch.stack(lista_tokens_selecionados, dim=0)
      #print("embedding_relevante.shape:",embedding_relevante.shape)

      # Calcula a média dos embeddings para os tokens de Si, removendo a primeira dimensão.
      # Entrada: <qtde_tokens> x <768 ou 1024>
      media_embedding_relevante = torch.mean(embedding_relevante, dim=0)
      # Saída: <768 ou 1024>
      #print("media_embedding_relevante.shape:", media_embedding_relevante.shape)
  else:
      media_embedding_relevante = None

  del embedding_documento
  del token_BERT_documento
  del documento
  del token_documento
  del pos_documento

  return media_embedding_relevante

### getEmbeddingDocumentoMean

Filtros:
- ALL - Sentença com todas as palavras
- NOUN - Sentença somente com substantivos
- VERB - Sentença somente com verbos
- VERB,NOUN - Sentença somente com verbos e substantivos

In [82]:
def getEmbeddingDocumentoMean(id_documento,
                              index_sentenca,
                              embedding_documento,
                              token_BERT_documento,
                              documento,
                              tokenizer,
                              token_documento,
                              pos_documento,
                              filtro=["ALL"]):
  '''
  Rediciona o cálculo da média dos embeddings de acordo com o filtro especificado.

  Parâmetros:
    `embedding_documento` - Embeddings do documento gerados pelo BERT.
    `token_BERT_documento` - Lista com os tokens do documento gerados pelo tokenizador BERT.
    `documento` - Texto com o documento.
    `tokenizer` - Tokenizador do BERT.
    `token_documento` - Lista com os tokens do documento.
    `pos_documento` - Lista com as POS-Tagging do documento.
    `filtro` - Filtro dos embeddings.
  '''

  if "ALL" in filtro:
    return getEmbeddingDocumentoComTodasPalavrasMean(embedding_documento)
  else:
    return getEmbeddingDocumentoRelevanteMean(id_documento,
                                              index_sentenca,
                                              embedding_documento,
                                              token_BERT_documento,
                                              documento,
                                              token_documento,
                                              pos_documento,
                                              filtro)

### getTokensEmbeddingsPOSSentenca
Gera os tokens, POS e embeddings de cada sentença.

In [83]:
# Dicionário de tokens de exceções e seus deslocamentos para considerar mais tokens do BERT em relação ao spaCy
# A tokenização do BERT gera mais tokens que a tokenização das palavras do spaCy
dic_excecao_maior = {"":-1,
                    }

In [84]:
def getExcecaoDicMaior(token, dic_excecao_maior):

  valor = dic_excecao_maior.get(token)
  if valor != None:
      return valor
  else:
      return -1

In [85]:
# Dicionário de tokens de exceções e seus deslocamentos para considerar menos tokens do BERT em relação ao spaCy
# A tokenização do BERT gera menos tokens que a tokenização das palavras do spaCy
dic_excecao_menor = {"1°":1,
                    }

In [86]:
def getExcecaoDicMenor(token, dic_excecao_menor):

  valor = dic_excecao_menor.get(token)
  if valor != None:
      return valor
  else:
      return -1

Função que retorna os embeddings, tokens e POS da sentença com um mesmo tamanho.

In [87]:
# Importa a biblioteca
import torch

def getTokensEmbeddingsPOSSentenca(embedding_documento,
                                   token_BERT_documento,
                                   sentenca):
  '''
  Retorna os tokens, as postagging e os embeddings dos tokens igualando a quantidade de tokens do spaCy com a tokenização do BERT de acordo com a estratégia.
  Usa a estratégia MEAN para calcular a média dos embeddings dos tokens que formam uma palavra.
  Usa a estratégia MAX para calcular o valor máximo dos embeddings dos tokens que formam uma palavra.
  '''

  #Guarda os tokens e embeddings
  lista_tokens = []
  lista_tokens_OOV = []
  lista_embeddings_MEAN = []
  lista_embeddings_MAX = []

  # Gera a tokenização e POS-Tagging da sentença
  sentenca_token, sentenca_pos = getListaTokensPOSSentenca(sentenca)

  # print("\nsentenca          :",sentenca)
  # print("sentenca_token      :",sentenca_token)
  # print("len(sentenca_token) :",len(sentenca_token))
  # print("sentenca_pos        :",sentenca_pos)
  # print("len(sentenca_pos)   :",len(sentenca_pos))

  # Recupera os embeddings da sentença dos embeddings do documento
  embedding_sentenca = embedding_documento
  sentenca_tokenizada_BERT = token_BERT_documento

  # embedding <qtde_tokens x 4096>
  # print("embedding_sentenca          :",embedding_sentenca.shape)
  # print("sentenca_tokenizada_BERT     :",sentenca_tokenizada_BERT)
  # print("len(sentenca_tokenizada_BERT):",len(sentenca_tokenizada_BERT))

  # Seleciona os pares de palavra a serem avaliadas
  pos_wi = 0 # Posição do token da palavra gerado pelo spaCy
  pos_wj = pos_wi # Posição do token da palavra gerado pelo BERT
  pos2 = -1

  # Enquanto o indíce da palavra pos_wj(2a palavra) não chegou ao final da quantidade de tokens do BERT
  while pos_wj < len(sentenca_tokenizada_BERT):

    # Seleciona os tokens da sentença
    wi = sentenca_token[pos_wi] # Recupera o token da palavra gerado pelo spaCy
    wi1 = ""
    pos2 = -1
    if pos_wi+1 < len(sentenca_token):
      wi1 = sentenca_token[pos_wi+1] # Recupera o próximo token da palavra gerado pelo spaCy

      # Localiza o deslocamento da exceção
      pos2 = getExcecaoDicMenor(wi+wi1, dic_excecao_menor)
      #print("Exceção pos2:", pos2)

    wj = sentenca_tokenizada_BERT[pos_wj] # Recupera o token da palavra gerado pelo BERT
    # print("wi[",pos_wi,"]=", wi)
    # print("wj[",pos_wj,"]=", wj)

    # Tratando exceções
    # Localiza o deslocamento da exceção
    pos = getExcecaoDicMaior(wi, dic_excecao_maior)
    #print("Exceção pos:", pos)

    if pos != -1 or pos2 != -1:
      if pos != -1:
        #print("Adiciona 1 Exceção palavra == wi or palavra = [UNK]:",wi)
        lista_tokens.append(wi)
        # Marca como fora do vocabulário do BERT
        lista_tokens_OOV.append(1)
        # Verifica se tem mais de um token
        if pos != 1:
          indice_token = pos_wj + pos
          #print("Calcula a média de :", pos_wj , "até", indice_token)
          embeddings_tokens_palavra = embedding_sentenca[pos_wj:indice_token]
          #print("embeddings_tokens_palavra:",embeddings_tokens_palavra.shape)
          # calcular a média dos embeddings dos tokens do BERT da palavra
          embedding_estrategia_MEAN = torch.mean(embeddings_tokens_palavra, dim=0)
          #print("embedding_estrategia_MEAN:",embedding_estrategia_MEAN.shape)
          lista_embeddings_MEAN.append(embedding_estrategia_MEAN)

          # calcular o máximo dos embeddings dos tokens do BERT da palavra
          embedding_estrategia_MAX, linha = torch.max(embeddings_tokens_palavra, dim=0)
          #print("embedding_estrategia_MAX:",embedding_estrategia_MAX.shape)
          lista_embeddings_MAX.append(embedding_estrategia_MAX)
        else:
          # Adiciona o embedding do token a lista de embeddings
          lista_embeddings_MEAN.append(embedding_sentenca[pos_wj])
          lista_embeddings_MAX.append(embedding_sentenca[pos_wj])

        # Avança para a próxima palavra e token do BERT
        pos_wi = pos_wi + 1
        pos_wj = pos_wj + pos
        #print("Proxima:")
        #print("wi[",pos_wi,"]=", sentenca_token[pos_wi])
        #print("wj[",pos_wj,"]=", sentenca_tokenizada_BERT[pos_wj])
      else:
        if pos2 != -1:
          #print("Adiciona 1 Exceção palavra == wi or palavra = [UNK]:",wi)
          lista_tokens.append(wi+wi1)
          # Marca como fora do vocabulário do BERT
          lista_tokens_OOV.append(1)
          # Verifica se tem mais de um token
          if pos2 == 1:
            # Adiciona o embedding do token a lista de embeddings
            lista_embeddings_MEAN.append(embedding_sentenca[pos_wj])
            lista_embeddings_MAX.append(embedding_sentenca[pos_wj])

          # Avança para a próxima palavra e token do BERT
          pos_wi = pos_wi + 2
          pos_wj = pos_wj + pos2
          #print("Proxima:")
          #print("wi[",pos_wi,"]=", sentenca_token[pos_wi])
          #print("wj[",pos_wj,"]=", sentenca_tokenizada_BERT[pos_wj])
    else:
      # Tokens iguais adiciona a lista, o token não possui subtoken
      if (wi == wj or wj=="[UNK]"):
        # Adiciona o token a lista de tokens
        #print("Adiciona 2 wi==wj or wj==[UNK]:", wi )
        lista_tokens.append(wi)
        # Marca como dentro do vocabulário do BERT
        lista_tokens_OOV.append(0)
        # Adiciona o embedding do token a lista de embeddings
        lista_embeddings_MEAN.append(embedding_sentenca[pos_wj])
        lista_embeddings_MAX.append(embedding_sentenca[pos_wj])
        #print("embedding1[pos_wj]:", embedding_sentenca[pos_wj].shape)
        # Avança para a próxima palavra e token do BERT
        pos_wi = pos_wi + 1
        pos_wj = pos_wj + 1

      else:
        # A palavra foi tokenizada pelo Wordpice com ## ou diferente do spaCy ou desconhecida
        # Inicializa a palavra a ser montada
        palavra_POS = wj
        indice_token = pos_wj + 1
        while  ((palavra_POS != wi) and indice_token < len(sentenca_tokenizada_BERT)):
            if "##" in sentenca_tokenizada_BERT[indice_token]:
              # Remove os caracteres "##" do token
              parte = sentenca_tokenizada_BERT[indice_token][2:]
            else:
              parte = sentenca_tokenizada_BERT[indice_token]

            palavra_POS = palavra_POS + parte
            #print("palavra_POS:",palavra_POS)
            # Avança para o próximo token do BERT
            indice_token = indice_token + 1

        #print("\nMontei palavra:",palavra_POS)
        if (palavra_POS == wi or palavra_POS == "[UNK]"):
            # Adiciona o token a lista
            #print("Adiciona 3 palavra == wi or palavra_POS = [UNK]:",wi)
            lista_tokens.append(wi)
            # Marca como fora do vocabulário do BERT
            lista_tokens_OOV.append(1)
            # Calcula a média dos tokens da palavra
            #print("Calcula o máximo :", pos_wj , "até", indice_token)
            embeddings_tokens_palavra = embedding_sentenca[pos_wj:indice_token]
            #print("embeddings_tokens_palavra2:",embeddings_tokens_palavra)
            #print("embeddings_tokens_palavra2:",embeddings_tokens_palavra.shape)

            # calcular a média dos embeddings dos tokens do BERT da palavra
            embedding_estrategia_MEAN = torch.mean(embeddings_tokens_palavra, dim=0)
            #print("embedding_estrategia_MEAN:",embedding_estrategia_MEAN)
            #print("embedding_estrategia_MEAN.shape:",embedding_estrategia_MEAN.shape)
            lista_embeddings_MEAN.append(embedding_estrategia_MEAN)

            # calcular o valor máximo dos embeddings dos tokens do BERT da palavra
            embedding_estrategia_MAX, linha = torch.max(embeddings_tokens_palavra, dim=0)
            #print("embedding_estrategia_MAX:",embedding_estrategia_MAX)
            #print("embedding_estrategia_MAX.shape:",embedding_estrategia_MAX.shape)
            lista_embeddings_MAX.append(embedding_estrategia_MAX)

        # Avança para o próximo token do spaCy
        pos_wi = pos_wi + 1
        # Pula para o próximo token do BERT
        pos_wj = indice_token

  # Verificação se as listas estão com o mesmo tamanho
  #if (len(lista_tokens) != len(sentenca_token)) or (len(lista_embeddings_MEAN) != len(sentenca_token)):
  if (len(lista_tokens) !=  len(lista_embeddings_MEAN)):
      print("\nsentenca                  :",sentenca)
      print("sentenca_pos              :",sentenca_pos)
      print("sentenca_token            :",sentenca_token)
      print("sentenca_tokenizada_BERT  :",sentenca_tokenizada_BERT)
      print("lista_tokens              :",lista_tokens)
      print("len(lista_tokens)         :",len(lista_tokens))
      print("lista_embeddings_MEAN     :",lista_embeddings_MEAN)
      print("len(lista_embeddings_MEAN):",len(lista_embeddings_MEAN))
      print("lista_embeddings_MAX      :",lista_embeddings_MAX)
      print("len(lista_embeddings_MAX) :",len(lista_embeddings_MAX))

  del embedding_sentenca
  del token_BERT_documento
  del sentenca_tokenizada_BERT
  del sentenca_token

  return lista_tokens, sentenca_pos, lista_tokens_OOV, lista_embeddings_MEAN, lista_embeddings_MAX

### Recupera detalhes do BERT

In [88]:
# Verifica o nome do modelo BERT a ser utilizado
MODELO_BERT = getNomeModeloBERT(model_args)

# Verifica o tamanho do modelo(default large)
TAMANHO_BERT = getTamanhoBERT(model_args)

# 5 - Projeção de embeddings

Apresenta os tokens gerados pelo BERT e seus embeddings.

## 5.1 Carregamento dos arquivos de dados

### 5.1.1 Especifica os nomes dos arquivos de dados



In [89]:
# Nome do arquivo
NOME_ARQUIVO_DATASET = "dataset.csv"
NOME_ARQUIVO_DATASET_COMPACTADO = "dataset.zip"
NOME_ARQUIVO_DATASET_POS = "datasetpos.csv"
NOME_ARQUIVO_DATASET_POS_COMPACTADO = "datasetpos.zip"

### 5.1.2 Cria o diretório local para receber os dados

In [90]:
# Importando as bibliotecas.
import os

# Se estiver executando no Google Colaboratory
if IN_COLAB:

  # Cria o diretório para receber os arquivos Originais e Permutados
  # Diretório a ser criado
  dirbase = DIRETORIO_LOCAL[:-1]

  if not os.path.exists(dirbase):
      # Cria o diretório
      os.makedirs(dirbase)
      logging.info("Diretório criado: {}".format(dirbase))
  else:
      logging.info("Diretório já existe: {}".format(dirbase))

INFO:root:Diretório já existe: /content/SRI


### 5.1.3 Copia os arquivos do Google Drive para o Colaboratory

In [91]:
# Se estiver executando no Google Colaboratory
if IN_COLAB:

  !cp "$DIRETORIO_DRIVE$NOME_ARQUIVO_DATASET_COMPACTADO" "$DIRETORIO_LOCAL"
  !cp "$DIRETORIO_DRIVE$NOME_ARQUIVO_DATASET_POS_COMPACTADO" "$DIRETORIO_LOCAL"

  logging.info("Terminei a cópia.")

INFO:root:Terminei a cópia.


Descompacta os arquivos.

Usa o unzip para descompactar:
*   `-o` sobrescreve o arquivo se existir
*   `-j` Não cria nenhum diretório
*   `-q` Desliga as mensagens
*   `-d` Diretório de destino


In [92]:
# Se estiver executando no Google Colaboratory
if IN_COLAB:
  !unzip -o -j -q "$DIRETORIO_LOCAL$NOME_ARQUIVO_DATASET_COMPACTADO" -d "$DIRETORIO_LOCAL"
  !unzip -o -j -q "$DIRETORIO_LOCAL$NOME_ARQUIVO_DATASET_POS_COMPACTADO" -d "$DIRETORIO_LOCAL"

  logging.info("Terminei a descompactação.")

INFO:root:Terminei a descompactação.


### 5.1.4 Carregamento das lista com os dados dos arquivos

#### Carrega o arquivo dos dados e POS

In [93]:
# Import das bibliotecas.
import pandas as pd

# Abre o arquivo e retorna o DataFrame
df_dataset = pd.read_csv(DIRETORIO_LOCAL + NOME_ARQUIVO_DATASET, sep=";", encoding="UTF-8")
df_dataset_pos = pd.read_csv(DIRETORIO_LOCAL + NOME_ARQUIVO_DATASET_POS, sep=";", encoding="UTF-8")

logging.info("TERMINADO DOCUMENTOS: {}.".format(len(df_dataset)))
logging.info("TERMINADO DOCUMENTOS POS: {}.".format(len(df_dataset_pos)))

INFO:root:TERMINADO DOCUMENTOS: 30.
INFO:root:TERMINADO DOCUMENTOS POS: 30.


In [94]:
#df_dataset = df_dataset[:5]

In [95]:
df_dataset.sample(5)

,id,sentencas,documento
28,29,['A Controladoria-Geral da União CGU realizará...,"A Controladoria-Geral da União CGU realizará, ..."
0,1,['O Brasil inicia o processo de elaboração do ...,O Brasil inicia o processo de elaboração do 5 ...
19,20,"['A Controladoria-Geral da União CGU publica, ...","A Controladoria-Geral da União CGU publica, ne..."
15,16,['O Instituto de Pesquisa Econômica Aplicada I...,O Instituto de Pesquisa Econômica Aplicada Ipe...
29,30,"['Durante cinco dias, profissionais que atuam ...","Durante cinco dias, profissionais que atuam em..."


In [96]:
df_dataset_pos.sample(5)

,id,pos_documento
4,5,"[[['A', 'Controladoria-Geral', 'da', 'União', ..."
21,22,"[[['A', 'Controladoria-Geral', 'da', 'União', ..."
0,1,"[[['O', 'Brasil', 'inicia', 'o', 'processo', '..."
19,20,"[[['A', 'Controladoria-Geral', 'da', 'União', ..."
8,9,"[[['Responsável', 'pela', 'elaboração', 'de', ..."


#### Corrigir os tipos de colunas dos dados e POS

Em dados originais:
- coluna 1 - `sentenças` carregadas do arquivo vem como string e não como lista.

Em dados originais pos:
- coluna 1 - `pos_documento` carregadas do arquivo vem como string e não como lista.

In [97]:
# Import das bibliotecas.
import ast # Biblioteca para conversão de string em lista

# Corrige os tipos dos dados
tipos = {"id": str}
df_dataset = df_dataset.astype(tipos)
df_dataset_pos = df_dataset_pos.astype(tipos)

# Verifica se o tipo da coluna não é list e converte
df_dataset["sentencas"] = df_dataset["sentencas"].apply(lambda x: ast.literal_eval(x) if type(x)!=list else x)

df_dataset_pos["pos_documento"] = df_dataset_pos["pos_documento"].apply(lambda x: ast.literal_eval(x) if type(x)!=list else x)

logging.info("TERMINADO CORREÇÃO ORIGINAIS: {}.".format(len(df_dataset)))
logging.info("TERMINADO CORREÇÃO ORIGINAIS POS: {}.".format(len(df_dataset_pos)))

INFO:root:TERMINADO CORREÇÃO ORIGINAIS: 30.
INFO:root:TERMINADO CORREÇÃO ORIGINAIS POS: 30.


#### Criando dados indexados

In [98]:
# Especifica o(s) campo(s) indexado(s) e faz uma cópia da lista indexada
df_dataset_indexado = df_dataset.set_index(["id"])
df_dataset_indexado.head()

,sentencas,documento
id,,
1,[O Brasil inicia o processo de elaboração do 5...,O Brasil inicia o processo de elaboração do 5 ...
2,[A Controladoria-Geral da União CGU apoia e pa...,A Controladoria-Geral da União CGU apoia e par...
3,[A Controladoria-Geral da União CGU convida os...,A Controladoria-Geral da União CGU convida os ...
4,[Artigo publicado na 22 Edição da Revista da C...,Artigo publicado na 22 Edição da Revista da CG...
5,[A Controladoria-Geral da União CGU realiza ch...,A Controladoria-Geral da União CGU realiza cha...


In [99]:
# Especifica o(s) campo(s) indexado(s) e faz uma cópia da lista indexada
df_dataset_pos_indexado = df_dataset_pos.set_index(["id"])
df_dataset_pos_indexado.head()

,pos_documento
id,
1,"[[[O, Brasil, inicia, o, processo, de, elabora..."
2,"[[[A, Controladoria-Geral, da, União, CGU, apo..."
3,"[[[A, Controladoria-Geral, da, União, CGU, con..."
4,"[[[Artigo, publicado, na, 22, Edição, da, Revi..."
5,"[[[A, Controladoria-Geral, da, União, CGU, rea..."


## 5.2 Gera os arquivos para o Embedding Projector

### 5.2.1 Cria o diretório para os arquivos

In [100]:
# Importando as bibliotecas.
import os

# Cria o diretório para receber os arquivos Originais e Permutados
# Diretório a ser criado
dirbase = DIRETORIO_LOCAL + "projector"

if not os.path.exists(dirbase):
  # Cria o diretório
  os.makedirs(dirbase)
  logging.info("Diretório criado: {}".format(dirbase))
else:
  logging.info("Diretório já existe: {}".format(dirbase))

INFO:root:Diretório criado: /content/SRI/projector


### 5.2.2 Gera os embeddings dos documentos

In [ ]:
# Import das bibliotecas.
from tqdm.notebook import tqdm as tqdm_notebook

lista_embeddings = []
lista_documentos = []
lista_documentos_id = []

# Se tem algum id no lista do filtro seleciona os documentos originais
if len(FILTRO_DO) != 0:
  documentos = df_dataset[df_dataset['id'].isin(FILTRO_DO)]

else:
  documentos = df_dataset

# Barra de progresso dos documentos
documentos_bar = tqdm_notebook(documentos.iterrows(), desc=f"Documentos", unit=f" documento", total=len(documentos))

# Percorre os documentos
for i, linha_documento in documentos_bar:

    # Recupera o id do documento
    id_documento = linha_documento[0]
    # print("id_documento:",id_documento)
    # print("linha_documento['documento']:", linha_documento['documento'])

    # Localiza a POSTagging do documento agrupado
    lista_pos_documento = df_dataset_pos_indexado.loc[id_documento][0]
    # print("lista_pos_documento:",lista_pos_documento)
    # print("len(lista_pos_documento):",len(lista_pos_documento))

    # Troca o documento por uma versão da concatenação das palavras geradas pelo spaCy
    # Percorre a lista_pos concatenando a posição 0 dos tokens
    documento_concatenado = " ".join(concatenaListas(lista_pos_documento, pos=0))
    # print("documento_concatenado:", documento_concatenado)
    documento = documento_concatenado

    # Recupera os embeddings do token CLS
    if ESTRATEGIA_EMBEDDING == 0:
        # print("Recuperando os embeddings do token CLS")
        # Gera embeddings concatenando as 4 últimas camadas do BERT
        token_embeddings, documento_tokenizado = getEmbeddingsUltimaCamada(documento, model, tokenizer)
        # print("token_embeddings=", token_embeddings.shape)

        #Recupera os embeddings do token CLS
        token_cls = token_embeddings[0:1]

        # Guarda os embeddings e os os outros dados do documento
        # Guarda o embedding do token [CLS]
        lista_embeddings.append(token_cls)
        lista_documentos.append(documento)
        lista_documentos_id.append(id_documento)

    else:
      # Recupera os embeddings da última camada
      if ESTRATEGIA_EMBEDDING == 1:
        # Gera embeddings concatenando da última camada do BERT
        token_embeddings, documento_tokenizado = getEmbeddingsUltimaCamada(documento, model, tokenizer)

        # Estratégia de medida Mean
        if ESTRATEGIA_POLLING == 0:
          # Entrada: <qtde_tokens> x <768 ou 1024>
          # print("token_embeddings=", token_embeddings.shape)
          # Remove o token [CLS] e [SEP]
          embedding_documento = torch.mean(token_embeddings[1:-1], dim=0)
          # Saída: <768 ou 1024>
          # print("embedding_documento=", embedding_documento.shape)
        else:
          # Estratégia de medida Max
          # Entrada: <qtde_tokens> x <768 ou 1024>
          # print("token_embeddings=", token_embeddings.shape)
          embedding_documento, linha = torch.max(token_embeddings[1:-1], dim=0)
          # Saída: <768 ou 1024>
          # print("embedding_documento=", embedding_documento.shape)

        # Guarda os embeddings e os os outros dados do documento
        lista_embeddings.append(embedding_documento)
        lista_documentos.append(documento)
        lista_documentos_id.append(id_documento)

      else:
        # Recupera os embeddings da concatenação das 4 últimas camadas
        if ESTRATEGIA_EMBEDDING == 2:
          # print("Recuperando a concatenação das 4 últimas camadas")
          # Gera embeddings concatenando as 4 últimas camadas do BERT
          token_embeddings, documento_tokenizado = getEmbeddingsConcat4UltimasCamadas(documento, model, tokenizer)

          # Estratégia de medida Mean
          if ESTRATEGIA_POLLING == 0:
            # Entrada: <qtde_tokens> x <768 ou 1024>
            # print("token_embeddings=", token_embeddings.shape)
            # Remove o token [CLS] e [SEP]
            embedding_documento = torch.mean(token_embeddings[1:-1], dim=0)
            # Saída: <768 ou 1024>
            # print("embedding_documento=", embedding_documento.shape)
          else:
            # Estratégia de medida Max
            # Entrada: <qtde_tokens> x <768 ou 1024>
            # print("token_embeddings=", token_embeddings.shape)
            embedding_documento, linha = torch.max(token_embeddings[1:-1], dim=0)
            # Saída: <768 ou 1024>
            # print("embedding_documento=", embedding_documento.shape)

          # Guarda os embeddings e os os outros dados do documento
          lista_embeddings.append(embedding_documento)
          lista_documentos.append(documento)
          lista_documentos_id.append(id_documento)


Documentos:   0%|          | 0/30 [00:00<?, ? documento/s]

/tmp/ipykernel_1814/1915413608.py:22: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  id_documento = linha_documento[0]
/tmp/ipykernel_1814/1915413608.py:27: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  lista_pos_documento = df_dataset_pos_indexado.loc[id_documento][0]


Mostra um documento processado.

In [ ]:
print(len(lista_embeddings[0]))
print(lista_documentos[0])

### 5.2.3 Gera os arquivos para o Embedding Projector

Gera o sufixo do nome do arquivo

In [ ]:
def getSufixoNomeArquivo():

  sufixo_arquivo = ""

  # Embeddingsdo token CLS
  if ESTRATEGIA_EMBEDDING == 0:
    # Tamanho dos embeddings
    sufixo_arquivo = sufixo_arquivo + "_" + str(lista_embeddings[0].size()[1]) + TAMANHO_BERT

    # Adiciona a palavra CLS
    sufixo_arquivo = sufixo_arquivo + "_CLS"

    # Não possui MEAN ou MAX
  else:
    # Embeddings da última camada ou da concatenação das 4 últimas camadas
    if ESTRATEGIA_EMBEDDING == 1 or ESTRATEGIA_EMBEDDING == 2:
      sufixo_arquivo = sufixo_arquivo + "_" + str(lista_embeddings[0].size()[0]) + TAMANHO_BERT

      # Média dos embeddings
      if ESTRATEGIA_POLLING == 0:
        sufixo_arquivo = sufixo_arquivo + "_MEAN"
      else:
        # Máximo dos embeddings
        sufixo_arquivo = sufixo_arquivo + "_MAX"
  return sufixo_arquivo

Arquivos com os valores dos embeddings

In [ ]:
# Import das bibliotecas.
from tqdm.notebook import tqdm as tqdm_notebook
import csv

# Recupera o sufixo do nome do arquivo
sufixo_arquivo = getSufixoNomeArquivo()
#print("sufixo_arquivo:", sufixo_arquivo)

# Concatena o diretório e o nome do arquivo
NOME_ARQUIVO_RECORD =  DIRETORIO_LOCAL + "projector/" + "records_documento" + sufixo_arquivo + ".tsv"

# Abre o arquivo
with open(NOME_ARQUIVO_RECORD, 'w', encoding='utf8') as tsvfile:
  # Cria um arquivo separado por tab
    writer = csv.writer(tsvfile, delimiter='\t')

    # Barra de progresso dos embedings
    lista_embeddings_bar = tqdm_notebook(enumerate(lista_embeddings), desc=f"Embeddings", unit=f" embedding", total=len(lista_embeddings))

    # Percorre os embeddings
    for i, documento_embedding in lista_embeddings_bar:

      # Embedding [CLS]
      if ESTRATEGIA_EMBEDDING == 0:

        # Converte os tensores em numpy array
        documento_embedding_np = documento_embedding.numpy()

        # Escreve no arquivo os embeddings do documento
        writer.writerows(documento_embedding_np)

      # Estratégia de medida Mean e Max
      else:

        # Converte os tensores em numpy array
        documento_embedding_np = [documento_embedding.numpy()]

        # Escreve no arquivo os embeddings do documento
        writer.writerows(documento_embedding_np)


Arquivo com os metadados dos embeddings

In [ ]:
# Import das bibliotecas.
from tqdm.notebook import tqdm as tqdm_notebook
import csv

# Recupera o sufixo do nome do arquivo
sufixo_arquivo = getSufixoNomeArquivo()
#print("sufixo_arquivo:", sufixo_arquivo)

# Concatena o diretório e o nome do arquivo
NOME_ARQUIVO_META =  DIRETORIO_LOCAL + "projector/" + "meta_documento" + sufixo_arquivo + ".tsv"

# Abre o arquivo
with open(NOME_ARQUIVO_META, 'w', encoding='utf8') as tsvfile:
    # Define o escritor do arquivo
    writer = csv.writer(tsvfile, delimiter='\t')

    writer.writerow(["Documento", "Id"])

    # Barra de progresso dos embedings
    lista_embeddings_bar = tqdm_notebook(enumerate(lista_embeddings), desc=f"Embeddings", unit=f" embedding", total=len(lista_embeddings))

    # Percorre os embeddings
    for i, documento_embedding in lista_embeddings_bar:
        #Escreva a sentença
        s = [lista_documentos[i],
             lista_documentos_id[i]]
        writer.writerow(s)

Faça o download dos arquivos `records_documento_4096.tsv` e `meta_documento_4096.tsv` e carregue em https://projector.tensorflow.org/ na opção load.

Faça o download dos arquivos gerados pelo notebook clicando na lateral esquerda no ícone "Arquivos".

Carrega os arquivos na ferramenta através do link "Load". Na opção existe um link botão para carregar o arquivo dos embeddings e um outro botão para carregar os metadados.

Você também pode utilizar um link a um arquivo de configuração config.json com a referência aos arquivos em algum repositório publico na internet, por exemplo github ou gist

Aqui um exemplo.

https://projector.tensorflow.org/?config=https://raw.githubusercontent.com/osmarbraz/cohebertv1projecao/main/config.json




### 5.3.4 Compacta e copia o arquivo do projetor para uma pasta do GoogleDrive

Compacta o arquivo gerado da comparação para facilitar o envio para o GoogleDrive

In [ ]:
# Nome do arquivo
NOME_ARQUIVO_PROJECTOR_COMPACTADO = "projector_documento.zip"

Compacta os arquivos.

Usa o zip para compactar:
*   `-r` Compacta o diretório
*   `-o` sobrescreve o arquivo se existir
*   `-j` Não cria nenhum diretório
*   `-q` Desliga as mensagens


In [ ]:
# Se estiver executando no Google Colaboratory
if IN_COLAB:

  !zip -r -o -q "$DIRETORIO_LOCAL$NOME_ARQUIVO_PROJECTOR_COMPACTADO" "$DIRETORIO_LOCAL""/projector/"

Copia o arquivo compactado para o GoogleDrive



In [ ]:
# Se estiver executando no Google Colaboratory
if IN_COLAB:

    # Copia o arquivo original
    !cp "$DIRETORIO_LOCAL$NOME_ARQUIVO_PROJECTOR_COMPACTADO" "$DIRETORIO_DRIVE"

    logging.info("Terminei a cópia")

## 5.3 Projeção dos embeddings

### Configuração

Verifica a versão do tensorflow

In [ ]:
try:
  # %tensorflow_version só existe no Colab.
  %tensorflow_version 2.x
except Exception:
  pass

%load_ext tensorboard

Importa a biblioteca

In [ ]:
# Importa de biblioteca
from tensorboard.plugins import projector

### Configura o diretório dos logs e arquivos de configuração




In [ ]:
# Importando as bibliotecas.
import os

# Configure um diretório de logs
log_dir = DIRETORIO_LOCAL + "/projector/"

if not os.path.exists(log_dir):
    os.makedirs(log_dir)

### Cria os arquivos de configuração dos embeddings

In [ ]:
# Configura o projetor
config = projector.ProjectorConfig()

# Configuração do primeiro conjunto de embeddings sem pooling
embedding = config.embeddings.add()

# Nome do tensor
if ESTRATEGIA_EMBEDDING == 1:
  embedding.tensor_name = "Embeddings Última camada BERTimbau"
else:
  embedding.tensor_name = "Embeddings Concat 4 últimas camadas BERTimbau"

# Caminho para os metadados
embedding.metadata_path = NOME_ARQUIVO_META
# Caminho para os tensores
embedding.tensor_path = NOME_ARQUIVO_RECORD
# Salva o arquivo de configuração
projector.visualize_embeddings(log_dir, config)

### Mata o processo

Se executar novamente o notebook é necessário matar o processo do tensorprojector.

In [ ]:
# Mata o processo do tensorboard
#!kill 407

### Visualizando a projeção

Na caixa de seleção selecione "PROJECTOR" no lugar de "INACTIVE".

Se o Embedding Projector não executar no notebook localmente utilize a versão online em https://projector.tensorflow.org/.
Clique no botão "**Load**" e faça upload do arquivo `meta_documento_XXX.tsv` no passo 1 (Step 1) e o arquivo `records_documento_XXX.txt` no passo 2(STep 2).
Os arquivos estão na pasta `./projector`.

In [ ]:
# Agora execute o tensorboard nos dados de log que acabamos de salvar.
%tensorboard --logdir $DIRETORIO_LOCAL/projector/

<IPython.core.display.Javascript object>

A busca com regex (Search /.*) deve utilizar a string de pesquisa entre `^` e `$`.